# Point 8 — Empty-Foreground Evaluation and Preprocessing Verification

This notebook reuses the dataset loaders, binary-mask conversion, AgriCLIP/DINO automatic-box generator, SAM2 model and checkpoint-loading functions from the Point 7 notebook. It evaluates the original test partitions of ReLeaf, WeedsGalore and EWI1S, and the established 20% VegAnn hold-out split.

It (1) audits image-mask dimensions and binary mask values, (2) counts empty and non-empty ground-truth masks, and (3) runs automatic AgriCLIP-box and full-image SAM2 inference on every empty-ground-truth image. No training is performed.

**Empty-mask scoring convention:** if both GT and prediction are empty, IoU and Dice are 1.0; if GT is empty but prediction contains foreground, IoU and Dice are 0.0. False-positive image rate and false-positive pixel area are reported separately.

Run cells in order. Paths and checkpoints use the same project-root assumptions as Point 7. The notebook does not fabricate empty samples: if a test partition has no empty masks, it reports zero and performs no empty-image inference for that dataset.

In [ ]:
# ============================================================
# PHASE 1 — MULTI-MASK SAM2 EXPERIMENT
# ============================================================

import os
import sys
import json
import random
import math
import copy
import time
import warnings
import inspect

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import cv2

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import (
    Sam2Model,
    Sam2Processor,
)

warnings.filterwarnings("ignore")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_NAME = "facebook/sam2.1-hiera-tiny"

# ------------------------------------------------------------
# AgriCLIP → SAM2 settings
# ------------------------------------------------------------

TOP_K = 32
BOX_MARGIN = 25
MASK_THRESHOLD = 0.5

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/zahranadeem/thesis"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "point7_target_failure_diagnostics"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("TOP_K:", TOP_K)
print("BOX_MARGIN:", BOX_MARGIN)
print("MASK_THRESHOLD:", MASK_THRESHOLD)
print("Output:", OUTPUT_ROOT)

In [ ]:
# ============================================================
# DATASET PATHS
# ============================================================

# ------------------------------------------------------------
# VegAnn SAM2 checkpoint
# ------------------------------------------------------------

VEGANN_SAM2_CHECKPOINT = (
    PROJECT_ROOT
    / "sam2_vegann_checkpoints"
    / "best_sam2_vegann_mask_decoder.pt"
)


# ------------------------------------------------------------
# ReLeaf
# ------------------------------------------------------------

RELEAF_ROOT = (
    PROJECT_ROOT / "CropAndWeedAndLeaf"
)

RELEAF_REPO = (
    PROJECT_ROOT / "releaf"
)

RELEAF_IMAGES_DIR = (
    RELEAF_ROOT / "images"
)

RELEAF_LABELS_DIR = (
    RELEAF_ROOT / "labels"
)

RELEAF_TEST_DIR = (
    RELEAF_REPO
    / "data"
    / "CWL"
    / "test"
)


# ------------------------------------------------------------
# WeedsGalore
# ------------------------------------------------------------

WEEDSGALORE_ROOT = (
    PROJECT_ROOT
    / "WeedsGalore_dataset"
)

WEEDS_IMAGE_DIR = (
    WEEDSGALORE_ROOT / "data"
)

WEEDS_MASK_DIR = (
    WEEDSGALORE_ROOT
    / "fields"
    / "ground_truth_semantic"
)


# ------------------------------------------------------------
# EWI1S
# ------------------------------------------------------------

EWIS1_ROOT = (
    PROJECT_ROOT / "EWIS1"
)


def find_nested_dataset_dir(
    root,
    dirname,
    extension=".png"
):

    candidates = []

    for p in root.rglob(dirname):

        if not p.is_dir():
            continue

        files = [
            x for x in p.iterdir()
            if x.is_file()
            and x.suffix.lower()
            == extension.lower()
        ]

        if len(files) > 0:
            candidates.append(
                (len(files), p)
            )

    if not candidates:
        return None

    candidates.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    return candidates[0][1]


EWIS1_IMAGE_DIR = find_nested_dataset_dir(
    EWIS1_ROOT,
    "images",
    ".png"
)

EWIS1_MASK_DIR = find_nested_dataset_dir(
    EWIS1_ROOT,
    "masks",
    ".png"
)


print("VegAnn checkpoint:", VEGANN_SAM2_CHECKPOINT)
print("ReLeaf images:", RELEAF_IMAGES_DIR)
print("ReLeaf labels:", RELEAF_LABELS_DIR)
print("WeedsGalore images:", WEEDS_IMAGE_DIR)
print("WeedsGalore masks:", WEEDS_MASK_DIR)
print("EWI1S images:", EWIS1_IMAGE_DIR)
print("EWI1S masks:", EWIS1_MASK_DIR)

In [ ]:
# ============================================================
# VERIFY PATHS
# ============================================================

paths_to_check = {
    "VegAnn checkpoint": VEGANN_SAM2_CHECKPOINT,
    "ReLeaf images": RELEAF_IMAGES_DIR,
    "ReLeaf labels": RELEAF_LABELS_DIR,
    "WeedsGalore images": WEEDS_IMAGE_DIR,
    "WeedsGalore masks": WEEDS_MASK_DIR,
    "EWI1S images": EWIS1_IMAGE_DIR,
    "EWI1S masks": EWIS1_MASK_DIR,
}

for name, path in paths_to_check.items():

    exists = (
        path is not None
        and path.exists()
    )

    print(
        f"{name:30s}: "
        f"{'OK' if exists else 'MISSING'}"
    )

if not VEGANN_SAM2_CHECKPOINT.exists():

    raise FileNotFoundError(
        f"VegAnn checkpoint not found:\n"
        f"{VEGANN_SAM2_CHECKPOINT}"
    )

In [ ]:
# ============================================================
# MASK UTILITIES
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".bmp"
}

MASK_EXTENSIONS = IMAGE_EXTENSIONS


def binary_mask_from_array(mask):

    if isinstance(
        mask,
        Image.Image
    ):
        mask = np.array(mask)

    elif torch.is_tensor(mask):

        mask = (
            mask
            .detach()
            .cpu()
            .numpy()
        )

    mask = np.asarray(mask)

    while mask.ndim > 2:
        mask = mask[..., 0]

    return (
        mask > 0
    ).astype(np.uint8)


def mask_to_box(mask):

    mask = binary_mask_from_array(
        mask
    )

    ys, xs = np.where(
        mask > 0
    )

    if len(xs) == 0:
        return None

    return np.array(
        [
            xs.min(),
            ys.min(),
            xs.max(),
            ys.max()
        ],
        dtype=np.int32
    )


def box_iou(
    box_a,
    box_b
):

    if box_a is None or box_b is None:
        return 0.0

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)

    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(
        0,
        ix2 - ix1
    )

    ih = max(
        0,
        iy2 - iy1
    )

    intersection = (
        iw * ih
    )

    area_a = (
        max(0, ax2 - ax1)
        * max(0, ay2 - ay1)
    )

    area_b = (
        max(0, bx2 - bx1)
        * max(0, by2 - by1)
    )

    union = (
        area_a
        + area_b
        - intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union

## 1. Target dataset loading and deterministic test splits

The data loaders and split logic below are reused from the source notebook. Unlike its validation-only cleanup cells, this diagnostic keeps empty test masks so they can be reported separately.

In [ ]:
# ============================================================
# RELEAF LOADER
# ============================================================

def load_releaf_image_mask(
    image_path
):

    image = Image.open(
        image_path
    ).convert("RGB")

    width, height = image.size

    mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    label_path = (
        RELEAF_LABELS_DIR
        / f"{image_path.stem}.txt"
    )

    if not label_path.exists():

        return image, mask

    with open(
        label_path,
        "r"
    ) as f:

        lines = f.readlines()

    for line in lines:

        values = (
            line
            .strip()
            .split()
        )

        if len(values) < 7:
            continue

        coords = [
            float(v)
            for v in values[1:]
        ]

        if len(coords) < 6:
            continue

        points = []

        for i in range(
            0,
            len(coords),
            2
        ):

            x = int(
                coords[i]
                * width
            )

            y = int(
                coords[i + 1]
                * height
            )

            points.append(
                [x, y]
            )

        points = np.asarray(
            points,
            dtype=np.int32
        )

        cv2.fillPoly(
            mask,
            [points],
            1
        )

    return image, mask

In [ ]:
# ============================================================
# RELEAF PAIRS
# ============================================================

releaf_pairs = []

releaf_image_files = sorted(
    [
        p
        for p in RELEAF_IMAGES_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)

for image_path in releaf_image_files:

    label_path = (
        RELEAF_LABELS_DIR
        / f"{image_path.stem}.txt"
    )

    if label_path.exists():

        releaf_pairs.append(
            {
                "image_path":
                    image_path,

                "label_path":
                    label_path,

                "dataset":
                    "ReLeaf"
            }
        )

print(
    "ReLeaf pairs:",
    len(releaf_pairs)
)

In [ ]:
#split Releadf 

releaf_official_test_names = set()

if RELEAF_TEST_DIR.exists():

    for p in RELEAF_TEST_DIR.iterdir():

        if (
            p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        ):
            releaf_official_test_names.add(
                p.name
            )

print(
    "Existing ReLeaf test images:",
    len(releaf_official_test_names)
)

if len(releaf_official_test_names) > 0:
    print(
        list(releaf_official_test_names)[:10]
    )

from pathlib import Path

releaf_official_test_names = {
    Path(name).name for name in releaf_official_test_names
}

releaf_official_test_names = {
    "image1.jpg", 
    "image2.jpg"
} 


def split_releaf_pairs(
    pairs,
    seed=42
):
    """  deterministic 80/10/10 split.
    """

    rng = random.Random(seed)

    pairs = list(pairs)

    official_test = []
    remaining = []

    for item in pairs:

        if (
            item["image_path"].name
            in releaf_official_test_names
        ):
            official_test.append(item)
        else:
            remaining.append(item)

    rng.shuffle(remaining)

    if len(official_test) > 0:

        split_point = int(
            0.90 * len(remaining)
        )

        train_pairs = (
            remaining[:split_point]
        )

        val_pairs = (
            remaining[split_point:]
        )

        test_pairs = official_test

        split_mode = (
            "Existing ReLeaf test preserved"
        )

    else:

        n = len(pairs)

        n_train = int(
            0.80 * n
        )

        n_val = int(
            0.10 * n
        )

        train_pairs = pairs[
            :n_train
        ]

        val_pairs = pairs[
            n_train:n_train + n_val
        ]

        test_pairs = pairs[
            n_train + n_val:
        ]

        split_mode = (
            "New deterministic 80/10/10 split"
        )

    return (
        train_pairs,
        val_pairs,
        test_pairs,
        split_mode
    )


releaf_train_pairs, releaf_val_pairs, releaf_test_pairs, releaf_split_mode = (
    split_releaf_pairs(
        releaf_pairs,
        seed=SEED
    )
)

print("ReLeaf split mode:")
print(releaf_split_mode)

print("\nTrain:", len(releaf_train_pairs))
print("Val  :", len(releaf_val_pairs))
print("Test :", len(releaf_test_pairs))

In [ ]:
# ============================================================
# WEEDSGALORE LOADER
# ============================================================

def find_mask_for_stem(
    mask_dir,
    stem
):

    matches = list(
        mask_dir.glob(
            f"{stem}.*"
        )
    )

    matches = [
        p
        for p in matches
        if p.suffix.lower()
        in MASK_EXTENSIONS
    ]

    if matches:
        return matches[0]

    return None


weedsgalore_pairs = []

weed_image_files = sorted(
    [
        p
        for p in WEEDS_IMAGE_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)

for image_path in weed_image_files:

    mask_path = find_mask_for_stem(
        WEEDS_MASK_DIR,
        image_path.stem
    )

    if mask_path is not None:

        weedsgalore_pairs.append(
            {
                "image_path":
                    image_path,

                "mask_path":
                    mask_path,

                "dataset":
                    "WeedsGalore"
            }
        )


def load_weedsgalore_image_mask(
    item
):

    image = Image.open(
        item["image_path"]
    ).convert("RGB")

    mask = Image.open(
        item["mask_path"]
    )

    mask = binary_mask_from_array(
        mask
    )

    return image, mask


print(
    "WeedsGalore pairs:",
    len(weedsgalore_pairs)
)

In [ ]:
# ============================================================
# WEEDSGALORE SPLIT
# ============================================================

def split_pairs(
    pairs,
    seed=42,
    train_fraction=0.80,
    val_fraction=0.10
):

    pairs = list(pairs)

    rng = random.Random(seed)

    rng.shuffle(pairs)

    n = len(pairs)

    n_train = int(
        train_fraction * n
    )

    n_val = int(
        val_fraction * n
    )

    train = pairs[
        :n_train
    ]

    val = pairs[
        n_train:
        n_train + n_val
    ]

    test = pairs[
        n_train + n_val:
    ]

    return train, val, test


(
    weedsgalore_train_pairs,
    weedsgalore_val_pairs,
    weedsgalore_test_pairs
) = split_pairs(
    weedsgalore_pairs,
    seed=SEED
)

print(
    "WeedsGalore train:",
    len(weedsgalore_train_pairs)
)

print(
    "WeedsGalore val:",
    len(weedsgalore_val_pairs)
)

print(
    "WeedsGalore test:",
    len(weedsgalore_test_pairs)
)

In [ ]:
# ============================================================
# EWI1S LOADER
# ============================================================

ewis1_pairs = []

ewis1_image_files = sorted(
    [
        p
        for p in EWIS1_IMAGE_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() == ".png"
    ]
)

for image_path in ewis1_image_files:

    mask_path = (
        EWIS1_MASK_DIR
        / image_path.name
    )

    if mask_path.exists():

        ewis1_pairs.append(
            {
                "image_path":
                    image_path,

                "mask_path":
                    mask_path,

                "dataset":
                    "EWI1S"
            }
        )


def load_ewis1_image_mask(
    item
):

    image = Image.open(
        item["image_path"]
    ).convert("RGB")

    mask_rgba = np.array(
        Image.open(
            item["mask_path"]
        ).convert("RGBA")
    )

    rgb = mask_rgba[:, :, :3]

    alpha = mask_rgba[:, :, 3]

    weed_pixels = (
        (rgb[:, :, 0] == 255)
        & (rgb[:, :, 1] == 127)
        & (rgb[:, :, 2] == 14)
        & (alpha > 0)
    )

    weed_mask = (
        weed_pixels
        .astype(np.uint8)
    )

    return image, weed_mask


print(
    "EWI1S pairs:",
    len(ewis1_pairs)
)

In [ ]:
# ============================================================
# EWI1S SPLIT
# ============================================================

(
    ewis1_train_pairs,
    ewis1_val_pairs,
    ewis1_test_pairs
) = split_pairs(
    ewis1_pairs,
    seed=SEED
)

print(
    "EWI1S train:",
    len(ewis1_train_pairs)
)

print(
    "EWI1S val:",
    len(ewis1_val_pairs)
)

print(
    "EWI1S test:",
    len(ewis1_test_pairs)
)

In [ ]:
# Build evaluation records from TEST partitions only.
TEST_DATASETS = {
    "ReLeaf": releaf_test_pairs,
    "WeedsGalore": weedsgalore_test_pairs,
    "EWI1S": ewis1_test_pairs,
}
print("TEST PARTITION SIZES")
for name, records in TEST_DATASETS.items():
    print(f"{name:15s}: {len(records)}")

## VegAnn hold-out partition
VegAnn is loaded using the same deterministic 80/20 split used in the earlier experiments. The 20% hold-out partition is treated as the evaluation partition for this diagnostic.

In [ ]:
# Load the same VegAnn dataset and deterministic split used in the earlier experiments
vegann = load_dataset('simonMadec/VegAnn')
vegann_split = vegann['train'].train_test_split(test_size=0.20, seed=SEED)
vegann_val_hf = vegann_split['test']
vegann_test_records = [{'hf_dataset': vegann_val_hf, 'index': i, 'dataset': 'VegAnn', 'id': f'vegann_{i:05d}'} for i in range(len(vegann_val_hf))]
TEST_DATASETS['VegAnn'] = vegann_test_records
print('VegAnn hold-out images:', len(vegann_test_records))
print('Evaluation partitions:', {k: len(v) for k, v in TEST_DATASETS.items()})

## 2. AgriCLIP/DINO automatic prompt generator

These cells reuse the same text-concept prompts, VegAnn-trained aligner, DINO Layer4 features and top-K merged-box implementation from the source notebook. No target-domain aligner is trained here.

In [ ]:
# ============================================================
# AGRICLIP SETUP
# ============================================================

AGRICLIP_PATH = (
    "/home/zahranadeem/thesis/"
    "agriclip_sam2/AgriCLIP/"
    "AgriCLIP_alignment"
)

if AGRICLIP_PATH not in sys.path:

    sys.path.insert(
        0,
        AGRICLIP_PATH
    )

from TextToConcept import (
    TextToConcept
)

from LinearAligner import (
    LinearAligner
)

print(
    "AgriCLIP modules imported."
)

In [ ]:
# ============================================================
# AGRICLIP ALIGNER
# ============================================================

ALIGNER_PATH = Path(
    "/home/zahranadeem/thesis/"
    "agriclip_sam2/AgriCLIP/"
    "Weights/Aligned_Models/"
    "Agri_Dino_aligner_DDEF_CDEF.pth"
)

print(
    "Aligner:",
    ALIGNER_PATH
)

print(
    "Exists:",
    ALIGNER_PATH.exists()
)

if not ALIGNER_PATH.exists():

    raise FileNotFoundError(
        f"Aligner not found:\n"
        f"{ALIGNER_PATH}"
    )

In [ ]:
# ============================================================
# DINO RESNET-50
# ============================================================

from torchvision import transforms


IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]


dino_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])


dino_model = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_resnet50",
    pretrained=True
)

dino_model.eval()

dino_model.to(
    DEVICE
)

for parameter in (
    dino_model.parameters()
):

    parameter.requires_grad = False


print(
    "DINO loaded."
)

In [ ]:
# ============================================================
# LOAD VEGANN AGRICLIP ALIGNER
# ============================================================

text_to_concept = TextToConcept(
    dino_model,
    "dino_resnet50"
)

text_to_concept.load_linear_aligner(
    str(ALIGNER_PATH)
)

aligner = (
    text_to_concept.linear_aligner
)

# aligner.eval() is not needed/available — LinearAligner isn't a torch.nn.Module,
# so it has no train()/eval() mode to toggle.

print(
    "AgriCLIP aligner loaded."
)

print(
    "W shape:",
    tuple(aligner.W.shape)
)

print(
    "b shape:",
    tuple(aligner.b.shape)
)

In [ ]:
# ============================================================
# DINO LAYER4 SPATIAL FEATURES
# ============================================================

captured_feature_map = None


def capture_layer4_output(
    module,
    inputs,
    output
):

    global captured_feature_map

    captured_feature_map = (
        output.detach()
    )


hook_handle = (
    dino_model.layer4
    .register_forward_hook(
        capture_layer4_output
    )
)

print(
    "DINO layer4 hook registered."
)

In [ ]:
# ============================================================
# EXTRACT SPATIAL DINO FEATURES
# ============================================================

@torch.no_grad()
def extract_dino_patch_features(
    pil_image
):

    global captured_feature_map

    captured_feature_map = None

    image_tensor = (
        dino_transform(
            pil_image
        )
        .unsqueeze(0)
        .to(DEVICE)
    )

    _ = dino_model(
        image_tensor
    )

    if captured_feature_map is None:

        raise RuntimeError(
            "DINO layer4 hook did not "
            "capture a feature map."
        )

    if captured_feature_map.ndim != 4:

        raise RuntimeError(
            "Expected feature map "
            "(B,C,H,W), got "
            f"{captured_feature_map.shape}"
        )

    (
        _,
        channels,
        grid_h,
        grid_w
    ) = captured_feature_map.shape

    patch_features = (
        captured_feature_map[0]
        .permute(1, 2, 0)
        .reshape(
            grid_h * grid_w,
            channels
        )
        .contiguous()
    )

    return (
        patch_features,
        grid_h,
        grid_w
    )

In [ ]:
# ============================================================
# ALIGN DINO PATCH FEATURES TO AGRICLIP SPACE
# ============================================================

@torch.no_grad()
def align_patch_features(
    patch_features
):

    patch_features = (
        patch_features
        .float()
        .to(DEVICE)
    )

    aligned_features = (
        aligner
        .get_aligned_representation(
            patch_features
        )
    )

    aligned_features = F.normalize(
        aligned_features,
        dim=1
    )

    return aligned_features

In [ ]:
# Vegetation text prompts and encoding

device = next(dino_model.parameters()).device

POSITIVE_PROMPTS = [

    "vegetation",
    "green vegetation",
    "crop plants",
    "crop vegetation",
    "agricultural plants",
    "green plants",
]

NEGATIVE_PROMPTS = [

    "soil",
    "bare soil",
    "ground",
    "bare ground",
    "non-vegetated ground",
    "agricultural soil",
]

print("Prompt strategy: POSITIVE / NEGATIVE")

print("\nPositive prompts:")
for i, prompt in enumerate(POSITIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")

print("\nNegative prompts:")
for i, prompt in enumerate(NEGATIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")


@torch.no_grad()
def encode_prompt_set(prompts):

    embeddings = []

    for prompt in prompts:

        embedding = text_to_concept.encode_text(
            [[prompt]]
        )

        if isinstance(embedding, np.ndarray):
            embedding = torch.from_numpy(
                embedding
            )

        if not torch.is_tensor(embedding):
            raise TypeError(
                "encode_text returned an unsupported type: "
                f"{type(embedding)}"
            )

        embedding = (
            embedding
            .detach()
            .float()
            .to(device)
            .reshape(-1)
        )

        embeddings.append(
            embedding
        )

    embeddings = torch.stack(
        embeddings,
        dim=0
    )

    # Normalize each prompt independently
    embeddings = F.normalize(
        embeddings,
        dim=1
    )

    # Average prompts
    mean_embedding = embeddings.mean(
        dim=0
    )

    # Normalize the averaged embedding
    mean_embedding = F.normalize(
        mean_embedding,
        dim=0
    )

    return embeddings, mean_embedding


# ------------------------------------------------------------
# POSITIVE
# ------------------------------------------------------------

positive_text_embeddings, positive_embedding = (
    encode_prompt_set(
        POSITIVE_PROMPTS
    )
)


# ------------------------------------------------------------
# NEGATIVE
# ------------------------------------------------------------

negative_text_embeddings, negative_embedding = (
    encode_prompt_set(
        NEGATIVE_PROMPTS
    )
)


print(
    "Positive embeddings:",
    positive_text_embeddings.shape
)

print(
    "Negative embeddings:",
    negative_text_embeddings.shape
)

print(
    "Positive mean embedding:",
    positive_embedding.shape
)

print(
    "Negative mean embedding:",
    negative_embedding.shape
)

# ============================================================
# POSITIVE - NEGATIVE CONCEPT
# ============================================================

vegetation_embedding = (
    positive_embedding
    - negative_embedding
)

vegetation_embedding = F.normalize(
    vegetation_embedding,
    dim=0
)

print(
    "Final vegetation embedding:",
    vegetation_embedding.shape
)

In [ ]:
# ============================================================
# AGRICLIP TOP-K → MERGED BOX
# ============================================================

@torch.no_grad()
def generate_agriclip_box(
    image,
    top_k=32,
    margin=25
):

    # --------------------------------------------------------
    # DINO patch features
    # --------------------------------------------------------

    (
        patch_features,
        grid_h,
        grid_w
    ) = extract_dino_patch_features(
        image
    )


    # --------------------------------------------------------
    # AgriCLIP alignment
    # --------------------------------------------------------

    aligned_features = (
        align_patch_features(
            patch_features
        )
    )


    # --------------------------------------------------------
    # Positive - Negative similarity
    # --------------------------------------------------------

    patch_scores = (
        aligned_features
        @ vegetation_embedding
    )

    scores = (
        patch_scores
        .cpu()
        .numpy()
    )


    # --------------------------------------------------------
    # Top-K
    # --------------------------------------------------------

    topk_indices = (
        np.argsort(scores)[::-1]
        [:top_k]
    )


    # --------------------------------------------------------
    # Image dimensions
    # --------------------------------------------------------

    W, H = image.size

    patch_h = (
        H / grid_h
    )

    patch_w = (
        W / grid_w
    )


    boxes = []


    # --------------------------------------------------------
    # Convert selected patches to boxes
    # --------------------------------------------------------

    for idx in topk_indices:

        r = (
            idx // grid_w
        )

        c = (
            idx % grid_w
        )

        x1 = int(
            c * patch_w
        )

        y1 = int(
            r * patch_h
        )

        x2 = int(
            (c + 1) * patch_w
        )

        y2 = int(
            (r + 1) * patch_h
        )

        boxes.append(
            [
                x1,
                y1,
                x2,
                y2
            ]
        )


    boxes = np.asarray(
        boxes,
        dtype=np.int32
    )


    # --------------------------------------------------------
    # MERGED BOX
    # --------------------------------------------------------

    xmin = boxes[:, 0].min()
    ymin = boxes[:, 1].min()

    xmax = boxes[:, 2].max()
    ymax = boxes[:, 3].max()


    # --------------------------------------------------------
    # Margin
    # --------------------------------------------------------

    xmin = max(
        0,
        xmin - margin
    )

    ymin = max(
        0,
        ymin - margin
    )

    xmax = min(
        W - 1,
        xmax + margin
    )

    ymax = min(
        H - 1,
        ymax + margin
    )


    merged_box = np.array(
        [
            xmin,
            ymin,
            xmax,
            ymax
        ],
        dtype=np.int32
    )


    return (
        merged_box,
        boxes,
        scores,
        topk_indices
    )


print(
    "generate_agriclip_box ready."
)

## 3. SAM2 model and dataset-specific decoder checkpoints

In [ ]:
# ============================================================
# SAM2 PROCESSOR
# ============================================================

sam2_processor = (
    Sam2Processor
    .from_pretrained(
        MODEL_NAME
    )
)

print(
    "SAM2 processor loaded."
)

In [ ]:
# ============================================================
# FRESH SAM2 MODEL
# ============================================================

def create_fresh_sam2():

    model = (
        Sam2Model
        .from_pretrained(
            MODEL_NAME
        )
    )

    model.to(
        DEVICE
    )

    model.eval()

    return model


print(
    "SAM2 model creator ready."
)

In [ ]:
# ============================================================
# DECODER CHECKPOINTS
# ============================================================

CHECKPOINTS = {

    "VegAnn_Baseline":
        str(
            PROJECT_ROOT
            / "sam2_vegann_checkpoints"
            / "best_sam2_vegann_mask_decoder.pt"
        ),

    "ReLeaf_Finetuned":
        str(
            PROJECT_ROOT
            / "sam2_domain_finetuning"
            / "VegAnn_to_ReLeaf"
            / "best_mask_decoder.pt"
        ),

    "WeedsGalore_Finetuned":
        str(
            PROJECT_ROOT
            / "sam2_domain_finetuning"
            / "VegAnn_to_WeedsGalore"
            / "best_mask_decoder.pt"
        ),

    "EWI1S_Finetuned":
        str(
            PROJECT_ROOT
            / "sam2_domain_finetuning"
            / "VegAnn_to_EWI1S"
            / "best_mask_decoder.pt"
        ),

    "Joint_Finetuned":
        str(
            PROJECT_ROOT
            / "sam2_domain_finetuning"
            / "VegAnn_ReLeaf_WeedsGalore_EWI1S"
            / "best_mask_decoder.pt"
        )
}


for name, path in CHECKPOINTS.items():

    print(
        f"{name}:"
    )

    print(
        path
    )

    print(
        "Exists:",
        os.path.exists(path)
    )

    print()

In [ ]:
# ============================================================
# LOAD DECODER ONLY
# ============================================================

def load_decoder_checkpoint(
    model,
    checkpoint_path
):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    if (
        isinstance(checkpoint, dict)
        and
        "mask_decoder_state_dict"
        in checkpoint
    ):

        state_dict = (
            checkpoint[
                "mask_decoder_state_dict"
            ]
        )

    else:

        state_dict = checkpoint


    model.mask_decoder.load_state_dict(
        state_dict,
        strict=True
    )

    model.eval()

    return checkpoint

In [ ]:
# ============================================================
# SINGLE-MASK SAM2
# CURRENT BASELINE
# ============================================================

@torch.no_grad()
def sam2_predict_single_mask(
    model,
    image,
    box,
    threshold=0.5
):

    inputs = sam2_processor(
        images=image,
        input_boxes=[
            [box.tolist()]
        ],
        return_tensors="pt"
    )

    inputs = {
        k: (
            v.to(DEVICE)
            if torch.is_tensor(v)
            else v
        )
        for k, v in inputs.items()
    }


    outputs = model(
        **inputs,
        multimask_output=False
    )


    logits = outputs.pred_masks[
        0,
        0,
        0
    ]


    probability = torch.sigmoid(
        logits
    )


    height = int(
        inputs[
            "original_sizes"
        ][0, 0].item()
    )

    width = int(
        inputs[
            "original_sizes"
        ][0, 1].item()
    )


    probability = F.interpolate(
        probability[None, None],
        size=(
            height,
            width
        ),
        mode="bilinear",
        align_corners=False
    )[0, 0]


    prediction = (
        probability >= threshold
    ).cpu().numpy().astype(
        np.uint8
    )


    confidence = float(
        outputs.iou_scores[
            0,
            0,
            0
        ].item()
    )


    return (
        prediction,
        probability.cpu().numpy(),
        confidence
    )

In [ ]:
# ============================================================
# SEGMENTATION METRICS
# ============================================================

def calculate_metrics(
    prediction,
    target
):

    prediction = (
        np.asarray(prediction)
        .astype(bool)
    )

    target = (
        np.asarray(target)
        .astype(bool)
    )


    if prediction.shape != target.shape:

        prediction = cv2.resize(
            prediction.astype(
                np.uint8
            ),
            (
                target.shape[1],
                target.shape[0]
            ),
            interpolation=
                cv2.INTER_NEAREST
        ).astype(bool)


    intersection = np.logical_and(
        prediction,
        target
    ).sum()

    union = np.logical_or(
        prediction,
        target
    ).sum()

    pred_area = prediction.sum()

    target_area = target.sum()


    iou = (
        intersection / union
        if union > 0
        else 0.0
    )


    dice = (
        2 * intersection
        /
        (pred_area + target_area)
        if (
            pred_area
            + target_area
        ) > 0
        else 0.0
    )


    tp = intersection

    fp = np.logical_and(
        prediction,
        ~target
    ).sum()

    fn = np.logical_and(
        ~prediction,
        target
    ).sum()


    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )


    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )


    return {

        "iou":
            float(iou),

        "dice":
            float(dice),

        "precision":
            float(precision),

        "recall":
            float(recall)
    }

In [ ]:
# ============================================================
# UNIFIED EVALUATION LOADER
# ============================================================

def load_evaluation_sample(
    record,
    dataset_name
):

    if dataset_name == "VegAnn":

        hf_dataset = (
            record["hf_dataset"]
        )

        index = (
            record["index"]
        )

        sample = (
            hf_dataset[index]
        )

        image = sample["image"]

        if not isinstance(
            image,
            Image.Image
        ):

            image = Image.fromarray(
                np.asarray(image)
            )

        image = image.convert(
            "RGB"
        )

        gt_mask = (
            binary_mask_from_array(
                sample["mask"]
            )
        )

        return (
            image,
            gt_mask
        )


    elif dataset_name == "ReLeaf":

        return load_releaf_image_mask(
            record["image_path"]
        )


    elif dataset_name == "WeedsGalore":

        return load_weedsgalore_image_mask(
            record
        )


    elif dataset_name == "EWI1S":

        return load_ewis1_image_mask(
            record
        )


    else:

        raise ValueError(
            f"Unknown dataset: "
            f"{dataset_name}"
        )

In [ ]:
# Point 7 checkpoint mapping: one target-specific checkpoint per target dataset.
POINT7_CHECKPOINTS = {
    "ReLeaf": CHECKPOINTS["ReLeaf_Finetuned"],
    "WeedsGalore": CHECKPOINTS["WeedsGalore_Finetuned"],
    "EWI1S": CHECKPOINTS["EWI1S_Finetuned"],
}
print("POINT 7 CHECKPOINT VERIFICATION")
for name, path in POINT7_CHECKPOINTS.items():
    print(f"{name:15s} | exists={Path(path).exists()} | {path}")
missing = [f"{k}: {v}" for k,v in POINT7_CHECKPOINTS.items() if not Path(v).exists()]
if missing:
    raise FileNotFoundError(
        "Missing target-specific checkpoint(s). Correct PROJECT_ROOT or checkpoint mapping before continuing:\n"
        + "\n".join(missing)
    )

## 1. Point 8 checkpoint mapping
Use the VegAnn baseline checkpoint for VegAnn and the corresponding fine-tuned checkpoint for each target dataset. The same checkpoint is used for both prompt types within a dataset.

In [ ]:
POINT8_CHECKPOINTS = {
    'VegAnn': CHECKPOINTS['VegAnn_Baseline'],
    'ReLeaf': CHECKPOINTS['ReLeaf_Finetuned'],
    'WeedsGalore': CHECKPOINTS['WeedsGalore_Finetuned'],
    'EWI1S': CHECKPOINTS['EWI1S_Finetuned'],
}
print('Checkpoint verification:')
for name, path in POINT8_CHECKPOINTS.items():
    print(f'{name:15s} exists={Path(path).exists()} | {path}')
missing = [f'{k}: {v}' for k, v in POINT8_CHECKPOINTS.items() if not Path(v).exists()]
if missing:
    raise FileNotFoundError('Missing checkpoint(s):\n' + '\n'.join(missing))

POINT8_OUTPUT = PROJECT_ROOT / 'point8_empty_foreground_evaluation'
POINT8_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Output directory:', POINT8_OUTPUT)

## 2. Dataset integrity and empty-mask scan
This scan covers every image in each evaluation partition, including empty masks. It records original image/mask dimensions, post-loader dimensions, binary-mask values, and empty status. Masks are resized only when necessary, using nearest-neighbour interpolation.

In [ ]:
def load_point8_sample(record, dataset_name):
    image, mask = load_evaluation_sample(record, dataset_name)
    image = image.convert('RGB')
    raw_mask = np.asarray(mask)
    mask = binary_mask_from_array(mask)
    original_mask_shape = tuple(mask.shape)
    resized = False
    if mask.shape != (image.height, image.width):
        mask = cv2.resize(mask.astype(np.uint8), (image.width, image.height), interpolation=cv2.INTER_NEAREST)
        resized = True
    mask = (mask > 0).astype(np.uint8)
    return image, mask, original_mask_shape, resized, raw_mask

integrity_rows = []
for dataset_name, records in TEST_DATASETS.items():
    for idx, record in enumerate(tqdm(records, desc=f'{dataset_name} integrity scan')):
        try:
            image, mask, original_shape, resized, raw_mask = load_point8_sample(record, dataset_name)
            sample_id = str(record.get('image_path', record.get('id', idx)))
            vals = np.unique(mask).tolist()
            integrity_rows.append({
                'dataset': dataset_name, 'image_id': sample_id,
                'image_width': image.width, 'image_height': image.height,
                'mask_original_height': original_shape[0], 'mask_original_width': original_shape[1],
                'dimensions_matched_after_resize': mask.shape == (image.height, image.width),
                'resized_nearest_neighbor': resized,
                'binary_values_valid': set(vals).issubset({0, 1}),
                'unique_binary_values': str(vals),
                'gt_vegetation_pixels': int(mask.sum()),
                'gt_is_empty': bool(mask.sum() == 0),
            })
        except Exception as exc:
            integrity_rows.append({'dataset': dataset_name, 'image_id': str(record.get('image_path', record.get('id', idx))), 'load_error': f'{type(exc).__name__}: {exc}'})

integrity_df = pd.DataFrame(integrity_rows)
integrity_df.to_csv(POINT8_OUTPUT / 'point8_dataset_integrity_per_image.csv', index=False)
display(integrity_df.groupby('dataset').agg(
    images_scanned=('image_id', 'count'),
    empty_images=('gt_is_empty', 'sum'),
    dimension_mismatches_after_resize=('dimensions_matched_after_resize', lambda x: int((~x).sum())),
    resized_masks=('resized_nearest_neighbor', 'sum'),
    invalid_binary_masks=('binary_values_valid', lambda x: int((~x).sum())),
).fillna(0))

empty_counts = integrity_df.groupby('dataset')['gt_is_empty'].agg(['count', 'sum']).rename(columns={'count':'evaluation_images', 'sum':'empty_images'})
empty_counts['non_empty_images'] = empty_counts['evaluation_images'] - empty_counts['empty_images']
display(empty_counts)
empty_counts.to_csv(POINT8_OUTPUT / 'point8_empty_image_counts.csv')

## 3. Empty-foreground inference
Only images with an empty converted GT mask are evaluated here. GT-derived boxes are not used. Each empty image is evaluated with the automatic AgriCLIP box and the full-image box using the same dataset checkpoint. False positives are measured at image and pixel levels.

In [ ]:
def score_empty_ground_truth(prediction, gt_mask):
    pred = np.asarray(prediction).astype(bool)
    gt = np.asarray(gt_mask).astype(bool)
    if gt.any():
        raise ValueError('score_empty_ground_truth expects an empty GT mask.')
    fp_pixels = int(pred.sum())
    total_pixels = int(pred.size)
    fp_area_fraction = fp_pixels / total_pixels if total_pixels else 0.0
    is_false_positive_image = bool(fp_pixels > 0)
    # Explicit convention: empty/empty = 1; empty/non-empty = 0.
    iou = 1.0 if fp_pixels == 0 else 0.0
    dice = 1.0 if fp_pixels == 0 else 0.0
    return {'iou': iou, 'dice': dice, 'false_positive_image': is_false_positive_image, 'false_positive_pixels': fp_pixels, 'false_positive_area_fraction': fp_area_fraction, 'false_positive_area_percent': 100.0 * fp_area_fraction}

@torch.no_grad()
def run_empty_inference_for_dataset(dataset_name, records):
    checkpoint_path = POINT8_CHECKPOINTS[dataset_name]
    model = create_fresh_sam2()
    load_decoder_checkpoint(model, checkpoint_path)
    print(f'\n{dataset_name}: loaded {checkpoint_path}')
    rows = []
    for idx, record in enumerate(tqdm(records, desc=f'{dataset_name} empty inference')):
        try:
            image, gt, _, _, _ = load_point8_sample(record, dataset_name)
            if gt.sum() != 0:
                continue
            h, w = gt.shape
            sample_id = str(record.get('image_path', record.get('id', idx)))
            auto_box, _, _, _ = generate_agriclip_box(image, top_k=TOP_K, margin=BOX_MARGIN)
            auto_box = clip_box_xyxy(auto_box, w, h)
            full_box = full_image_box(w, h)
            for prompt_name, box in [('agriclip_auto', auto_box), ('full_image', full_box)]:
                pred, prob, confidence = sam2_predict_single_mask(model, image, box, threshold=MASK_THRESHOLD)
                scores = score_empty_ground_truth(pred, gt)
                rows.append({
                    'dataset': dataset_name, 'image_id': sample_id, 'prompt': prompt_name,
                    'image_width': w, 'image_height': h, 'checkpoint': str(checkpoint_path),
                    'sam2_confidence': float(confidence),
                    'prompt_area_fraction': float(box_area_fraction(box, w, h)),
                    **scores,
                })
        except Exception as exc:
            print(f'WARNING {dataset_name} image {idx}: {type(exc).__name__}: {exc}')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return rows

empty_inference_rows = []
for dataset_name, records in TEST_DATASETS.items():
    empty_records = []
    for idx, record in enumerate(records):
        try:
            _, gt, _, _, _ = load_point8_sample(record, dataset_name)
            if gt.sum() == 0: empty_records.append(record)
        except Exception: pass
    print(f'{dataset_name}: {len(empty_records)} empty images found.')
    if empty_records:
        empty_inference_rows.extend(run_empty_inference_for_dataset(dataset_name, empty_records))

empty_inference_df = pd.DataFrame(empty_inference_rows)
empty_inference_df.to_csv(POINT8_OUTPUT / 'point8_empty_inference_per_image.csv', index=False)
display(empty_inference_df.head())
print('Per-image results saved:', POINT8_OUTPUT / 'point8_empty_inference_per_image.csv')

## 4. Summary and exported results
False-positive image rate is the fraction of empty GT images with at least one predicted foreground pixel. Mean false-positive area is the average percentage of pixels predicted as vegetation across empty images, including zero-prediction images.

In [ ]:
if empty_inference_df.empty:
    print('No empty GT images were found in the evaluated partitions; no empty-image SAM2 inference was required.')
    empty_summary = pd.DataFrame(columns=['dataset','prompt','empty_images','false_positive_images','false_positive_image_rate_percent','mean_false_positive_pixels','mean_false_positive_area_percent','mean_empty_iou','mean_empty_dice'])
else:
    empty_summary = empty_inference_df.groupby(['dataset','prompt']).agg(
        empty_images=('image_id','count'),
        false_positive_images=('false_positive_image','sum'),
        false_positive_image_rate_percent=('false_positive_image', lambda x: 100.0 * x.mean()),
        mean_false_positive_pixels=('false_positive_pixels','mean'),
        mean_false_positive_area_percent=('false_positive_area_percent','mean'),
        mean_empty_iou=('iou','mean'),
        mean_empty_dice=('dice','mean'),
    ).reset_index().round(4)
display(empty_summary)
empty_summary.to_csv(POINT8_OUTPUT / 'point8_empty_inference_summary.csv', index=False)
print('\nAll Point 8 outputs saved to:', POINT8_OUTPUT)
print('1. point8_dataset_integrity_per_image.csv')
print('2. point8_empty_image_counts.csv')
print('3. point8_empty_inference_per_image.csv')
print('4. point8_empty_inference_summary.csv')